# Report figures (from saved tables, no GPU)

Reads `accuracy.csv`, `lens.csv` and `status.csv` written by `run_pipeline.ipynb` and draws every figure. When real data or adapters land, rerun the pipeline and then this notebook. Only the config cell changes.

In [ ]:
# The project folder (with xling/, data/, notebooks/) lives in Google Drive.
# Change PROJECT_DIR if you put it somewhere else, or replace this cell with a `git clone`.
from google.colab import drive
drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/64610"
%cd {PROJECT_DIR}
import sys
sys.path.insert(0, PROJECT_DIR)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from xling import plots
from xling.evaluate import summarize

# Fictional updates: round2 judged against round1.
OUT, EDIT, REF, STAGES = "results/fictional", "round2", "round1", ("base", "round1", "round2")
# Toy counterfactuals instead:
# OUT, EDIT, REF, STAGES = "results/toy", "toy", "base", None

acc    = pd.read_csv(f"{OUT}/accuracy.csv")
lens   = pd.read_csv(f"{OUT}/lens.csv")
status = pd.read_csv(f"{OUT}/status.csv")
print(open(f"{OUT}/run_info.json").read()[:600])

## Retrieval accuracy
`summarize` gives one row per trained model and language. For fictional facts, the `round1` rows say whether the *original* fact was learned in English and reached each language. The `round2` rows say the same for the update, counting only prompts where round 1 had worked (others are `ref_unknown`).

In [ ]:
display(summarize(status))
import os
if STAGES and os.path.exists("adapters/fictional/history.csv"):
    plots.plot_training_history(pd.read_csv("adapters/fictional/history.csv")); plt.show()
if STAGES:
    plots.plot_stages(acc, STAGES); plt.show()
for metric in ["top1_old", "top1_new", "full_prefers_new"]:
    print(metric); display(plots.accuracy_table(acc, metric).style.format("{:.0%}"))
plots.plot_accuracy(acc, EDIT, REF, metric="top1_new"); plt.show()
plots.plot_accuracy(acc, EDIT, REF, metric="full_prefers_new"); plt.show()

## Logit lens

In [ ]:
plots.plot_D_by_layer(lens, EDIT, REF); plt.show()
plots.plot_delta_D(lens, EDIT, REF); plt.show()

## Successful vs failed transfer
The proposal's main comparison: per target language, the D curve for prompts where the update transferred vs where it didn't (English shown dashed). With few facts these curves are anecdotes.

In [ ]:
st = status[status.model == EDIT]
for make in (lambda: plots.plot_D_by_status(lens, st, EDIT), lambda: plots.plot_onset(st, EDIT), lambda: plots.plot_profile(st, EDIT)):
    try:
        make(); plt.show()
    except ValueError as e:
        print("skipped:", e)

In [ ]:
# Save PNGs for the report draft.
print(plots.save_all(acc, lens, status, OUT, EDIT, REF, STAGES))